# Transfermarkt national teams scraper

This notebook downloads national-team squad age and market value data from Transfermarkt and saves it as a CSV file.

In [1]:
# Install the packages needed for browser scraping and HTML parsing.
%pip install -q pandas beautifulsoup4 playwright
!python -m playwright install chromium

Note: you may need to restart the kernel to use updated packages.


## Setup

Imports libraries and sets the Transfermarkt URL, output file and page range.

In [2]:
# Import libraries and define scraping settings.
import re
from pathlib import Path
from urllib.parse import urlencode

import pandas as pd
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeoutError


BASE_URL = "https://www.transfermarkt.com/vereins-statistik/wertvollstenationalmannschaften/marktwertetop"

OUTPUT_CSV = "transfermarkt_national_teams.csv"

START_PAGE = 1
END_PAGE = 10

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/120.0.0.0 Safari/537.36"
)

In [3]:
# Helper functions for building URLs, cleaning values and parsing Transfermarkt tables.
def build_url(page_number):
    # Builds the URL for one Transfermarkt result page.
    params = {
        "kontinent_id": 0,
        "plus": 1,
        "page": page_number,
    }
    return f"{BASE_URL}?{urlencode(params)}"


def clean_text(x):
    # Removes extra spaces and converts empty values to missing values.
    if x is None:
        return pd.NA

    x = re.sub(r"\s+", " ", str(x)).strip()

    if x in {"", "-", "–", "—"}:
        return pd.NA

    return x


def clean_country_cell(td):
    link = td.select_one("a[href*='/nationalmannschaft/']")
    if link:
        return clean_text(link.get_text(" ", strip=True))

    return clean_text(td.get_text(" ", strip=True))


def parse_money_to_eur(value):
    # Converts values such as €1.2bn, €500m or €750k into numeric euros.
    if pd.isna(value):
        return pd.NA

    text = str(value).strip()
    if text in {"", "-", "–", "—"}:
        return pd.NA

    text = (
        text.replace("€", "")
            .replace("EUR", "")
            .replace(",", "")
            .strip()
            .lower()
    )

    multiplier = 1

    if text.endswith("bn"):
        multiplier = 1_000_000_000
        text = text[:-2]
    elif text.endswith("m"):
        multiplier = 1_000_000
        text = text[:-1]
    elif text.endswith("k"):
        multiplier = 1_000
        text = text[:-1]

    try:
        return float(text) * multiplier
    except ValueError:
        return pd.NA


def parse_int(value):
    if pd.isna(value):
        return pd.NA
    text = str(value)
    match = re.search(r"\d+", text)
    return int(match.group(0)) if match else pd.NA


def parse_float(value):
    if pd.isna(value):
        return pd.NA
    try:
        return float(str(value).replace(",", "."))
    except ValueError:
        return pd.NA


def parse_transfermarkt_table(html, page_number, debug=False):
    # Reads one HTML table and converts it into a clean dataframe.
    soup = BeautifulSoup(html, "html.parser")

    table = soup.select_one("table.items")
    if table is None:
        raise ValueError(f"The table table.items was not found on page {page_number}.")

    rows = table.select("tbody > tr.odd, tbody > tr.even")
    data = []

    for row in rows:
        tds = row.find_all("td", recursive=False)

        if len(tds) < 7:
            continue

        item = {
            "rank": clean_text(tds[0].get_text(" ", strip=True)),
            "country": clean_country_cell(tds[1]),
            "confederation": clean_text(tds[2].get_text(" ", strip=True)),
            "squad_size": clean_text(tds[3].get_text(" ", strip=True)),
            "avg_age": clean_text(tds[4].get_text(" ", strip=True)),
            "market_value": clean_text(tds[5].get_text(" ", strip=True)),
            "avg_market_value_of_players": clean_text(tds[6].get_text(" ", strip=True)),
            "page": page_number,
        }

        data.append(item)

    df = pd.DataFrame(data)

    if df.empty:
        return df

    df["rank"] = df["rank"].map(parse_int).astype("Int64")
    df["squad_size"] = df["squad_size"].map(parse_int).astype("Int64")
    df["avg_age"] = df["avg_age"].map(parse_float).astype("Float64")

    df["market_value_eur"] = df["market_value"].map(parse_money_to_eur).astype("Float64")
    df["avg_market_value_of_players_eur"] = (
        df["avg_market_value_of_players"].map(parse_money_to_eur).astype("Float64")
    )

    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = df[col].map(clean_text)

    df = df.drop_duplicates(subset=["rank", "country"], keep="first")
    df = df.sort_values("rank").reset_index(drop=True)

    if debug:
        print(f"Page {page_number}: parsed rows = {len(df)}")

    return df


async def accept_cookies_or_close_popups(page):
    # Closes cookie banners or popups that can block the table.
    selectors = [
        "button:has-text('Accept')",
        "button:has-text('Accept all')",
        "button:has-text('AGREE')",
        "button:has-text('Agree')",
        "button:has-text('I Accept')",
        "button:has-text('Consent')",
        "button:has-text('Continue')",
        "text=Accept all",
        "text=AGREE",
    ]

    for selector in selectors:
        try:
            loc = page.locator(selector)
            if await loc.count() > 0:
                await loc.first.click(timeout=3000, force=True)
                await page.wait_for_timeout(1000)
        except Exception:
            pass

    close_selectors = [
        "button[aria-label='Close']",
        "button:has-text('Close')",
        ".modal-close",
        ".close",
        "#google_vignette button",
    ]

    for selector in close_selectors:
        try:
            loc = page.locator(selector)
            if await loc.count() > 0:
                await loc.first.click(timeout=2000, force=True)
                await page.wait_for_timeout(500)
        except Exception:
            pass


async def fetch_transfermarkt_page(page_number, context, debug=False):
    # Opens one page in the browser and parses its table.
    url = build_url(page_number)
    page = await context.new_page()

    print(f"Loading page {page_number}: {url}")

    try:
        await page.goto(url, wait_until="domcontentloaded", timeout=60000)
    except PlaywrightTimeoutError:
        print(f"page.goto timeout on page {page_number}, continuing.")
    except Exception as e:
        print(f"page.goto issue on page {page_number}: {e}")

    await page.wait_for_timeout(3000)
    await accept_cookies_or_close_popups(page)

    try:
        await page.wait_for_selector("table.items tbody tr", timeout=30000)
    except Exception:
        print(f"The table on page {page_number} did not load within the time limit.")

    html = await page.content()
    await page.close()

    df_page = parse_transfermarkt_table(html, page_number, debug=debug)

    print(f"Page {page_number}: {len(df_page)} rows")
    return df_page


async def scrape_transfermarkt_all_pages(start_page=START_PAGE, end_page=END_PAGE, headless=True, debug=True):
    # Loops through all selected pages and combines them into one dataframe.
    all_pages = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=headless,
            args=[
                "--disable-blink-features=AutomationControlled",
                "--no-sandbox",
                "--disable-dev-shm-usage",
            ],
        )

        context = await browser.new_context(
            viewport={"width": 1440, "height": 1000},
            user_agent=USER_AGENT,
            extra_http_headers={
                "Accept-Language": "en-US,en;q=0.9",
                "Referer": "https://www.transfermarkt.com/",
            },
        )

        for page_number in range(start_page, end_page + 1):
            try:
                df_page = await fetch_transfermarkt_page(
                    page_number=page_number,
                    context=context,
                    debug=debug,
                )

                all_pages.append(df_page)

                page_wait = await context.new_page()
                await page_wait.wait_for_timeout(1000)
                await page_wait.close()

            except Exception as e:
                print(f"Error on page {page_number}: {e}")

        await browser.close()

    if not all_pages:
        return pd.DataFrame()

    df = pd.concat(all_pages, ignore_index=True)
    df = df.drop_duplicates(subset=["rank", "country"], keep="first")

    if "rank" in df.columns:
        df = df.sort_values("rank").reset_index(drop=True)

    return df


def save_transfermarkt_data(df):
    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    print(f"Saved: {OUTPUT_CSV}")
    return df

## Run scraper

Runs the scraper over the selected page range, saves the CSV and shows a basic preview.

In [4]:
# Download and save Transfermarkt data.
df_raw = await scrape_transfermarkt_all_pages(
    start_page=START_PAGE,
    end_page=END_PAGE,
    headless=True,
    debug=True,
)

if df_raw.empty:
    print("Could not retrieve data.")
    print("Try running: df_raw = await scrape_transfermarkt_all_pages(headless=False, debug=True)")
else:
    df = save_transfermarkt_data(df_raw)
    display(df.head(20))
    display(df.tail(20))
    print(f"Number of retrieved rows: {len(df)}")

Loading page 1: https://www.transfermarkt.com/vereins-statistik/wertvollstenationalmannschaften/marktwertetop?kontinent_id=0&plus=1&page=1
Page 1: parsed rows = 25
Page 1: 25 rows
Loading page 2: https://www.transfermarkt.com/vereins-statistik/wertvollstenationalmannschaften/marktwertetop?kontinent_id=0&plus=1&page=2
Page 2: parsed rows = 25
Page 2: 25 rows
Loading page 3: https://www.transfermarkt.com/vereins-statistik/wertvollstenationalmannschaften/marktwertetop?kontinent_id=0&plus=1&page=3
Page 3: parsed rows = 25
Page 3: 25 rows
Loading page 4: https://www.transfermarkt.com/vereins-statistik/wertvollstenationalmannschaften/marktwertetop?kontinent_id=0&plus=1&page=4
Page 4: parsed rows = 25
Page 4: 25 rows
Loading page 5: https://www.transfermarkt.com/vereins-statistik/wertvollstenationalmannschaften/marktwertetop?kontinent_id=0&plus=1&page=5
Page 5: parsed rows = 25
Page 5: 25 rows
Loading page 6: https://www.transfermarkt.com/vereins-statistik/wertvollstenationalmannschaften/mark

,rank,country,confederation,squad_size,avg_age,market_value,avg_market_value_of_players,page,market_value_eur,avg_market_value_of_players_eur
0,1,France,UEFA,26,27.0,€1.52bn,€58.58m,1,1520000000.0,58580000.0
1,2,England,UEFA,26,27.2,€1.36bn,€52.43m,1,1360000000.0,52430000.0
2,3,Spain,UEFA,26,26.8,€1.22bn,€47.03m,1,1220000000.0,47030000.0
3,4,Portugal,UEFA,26,28.1,€1.01bn,€38.67m,1,1010000000.0,38670000.0
4,5,Germany,UEFA,26,28.1,€947.00m,€36.42m,1,947000000.0,36420000.0
5,6,Brazil,South American Football Confederation,26,29.4,€928.20m,€35.70m,1,928200000.0,35700000.0
6,7,Argentina,South American Football Confederation,25,29.1,€782.50m,€31.30m,1,782500000.0,31300000.0
7,8,Netherlands,UEFA,26,27.8,€754.20m,€29.01m,1,754200000.0,29010000.0
8,9,Norway,UEFA,26,26.8,€589.90m,€22.69m,1,589900000.0,22690000.0
9,10,Belgium,UEFA,26,27.6,€547.50m,€21.06m,1,547500000.0,21060000.0


,rank,country,confederation,squad_size,avg_age,market_value,avg_market_value_of_players,page,market_value_eur,avg_market_value_of_players_eur
228,229,Christmas Island,previous teams,0,<NA>,<NA>,<NA>,10,<NA>,<NA>
229,230,Serbia and Montenegro,previous teams,0,<NA>,<NA>,<NA>,10,<NA>,<NA>
230,231,Nauru,Non-FIFA members,0,<NA>,<NA>,<NA>,10,<NA>,<NA>
231,232,Palau,Non-FIFA members,0,<NA>,<NA>,<NA>,10,<NA>,<NA>
232,233,Zaire,previous teams,0,<NA>,<NA>,<NA>,10,<NA>,<NA>
233,234,Dutch East Indies,previous teams,0,<NA>,<NA>,<NA>,10,<NA>,<NA>
234,235,Marshall Islands,Non-FIFA members,22,25.4,<NA>,<NA>,10,<NA>,<NA>
235,236,Northern Mariana Islands,AFC,0,<NA>,<NA>,<NA>,10,<NA>,<NA>
236,237,Niue,OFC,0,<NA>,<NA>,<NA>,10,<NA>,<NA>
237,238,Saar,previous teams,0,<NA>,<NA>,<NA>,10,<NA>,<NA>


Number of retrieved rows: 248


## Check saved CSV

Reads the saved file back into Python to confirm that the export worked.

In [5]:
# Load the saved Transfermarkt file as a final check.
df_check = pd.read_csv(OUTPUT_CSV)
df_check.head()

,rank,country,confederation,squad_size,avg_age,market_value,avg_market_value_of_players,page,market_value_eur,avg_market_value_of_players_eur
0,1,France,UEFA,26,27.0,€1.52bn,€58.58m,1,1.520000e+09,58580000.0
1,2,England,UEFA,26,27.2,€1.36bn,€52.43m,1,1.360000e+09,52430000.0
2,3,Spain,UEFA,26,26.8,€1.22bn,€47.03m,1,1.220000e+09,47030000.0
3,4,Portugal,UEFA,26,28.1,€1.01bn,€38.67m,1,1.010000e+09,38670000.0
4,5,Germany,UEFA,26,28.1,€947.00m,€36.42m,1,9.470000e+08,36420000.0
